# NullVector Progress Notebook
## major-changes-v2 Phase F-J Retrieval Core Verification

Purpose: verify the retrieval corpus, deterministic query planning, page-scoped visual retrieval, and downstream QA adapter behavior.

Prerequisites: the repository dev environment from `pyproject.toml` must be installed so the notebook can import `nullvector` and execute the synthetic fixture helpers.


### Environment
- Runs entirely against synthetic local artifacts under `notebooks/_artifacts/retrieval-progress/`.
- Does not require network access or live multimodal credentials.
- Fails on deprecation warnings to match repository validation policy.


In [ ]:
# environment setup
from pathlib import Path
import json
import shutil
import sys
import warnings

warnings.filterwarnings("error", category=DeprecationWarning)

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

ARTIFACT_ROOT = REPO_ROOT / "notebooks" / "_artifacts" / "retrieval-progress"
BUNDLE_ROOT = ARTIFACT_ROOT / "bundle"
shutil.rmtree(BUNDLE_ROOT, ignore_errors=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print({"repo_root": str(REPO_ROOT), "artifact_root": str(ARTIFACT_ROOT)})


In [ ]:
# imports
from nullvector.retrieval import (
    QueryPlanner,
    RetrievalCorpusBuilder,
    RetrievalQAService,
    RetrievalRanker,
    RetrievalService,
    augment_corpus_with_attachments,
    load_retrieval_corpus,
    load_retrieval_manifest,
)
from tests.retrieval.support import write_synthetic_bundle


In [ ]:
# configuration
VISUAL_QUERY = "what is the image on first page about?"
TEXT_QUERY = "Alpha body line"
bundle = write_synthetic_bundle(BUNDLE_ROOT)
retrieval_service = RetrievalService(QueryPlanner(), RetrievalRanker())
qa_service = RetrievalQAService(retrieval_service)


In [ ]:
# execution
retrieval_manifest = RetrievalCorpusBuilder().build(
    acquisition_manifest_path=str(bundle.acquisition_manifest_path),
    tree_manifest_path=str(bundle.tree_manifest_path),
)
loaded_manifest = load_retrieval_manifest(retrieval_manifest.artifact_root + "/manifest.json")
corpus = load_retrieval_corpus(loaded_manifest.corpus_path)
augmented_corpus = augment_corpus_with_attachments(
    corpus=corpus,
    attachments=(bundle.cached_attachment,),
)
visual_hits = retrieval_service.search(corpus=corpus, query=VISUAL_QUERY, limit=3)
visual_response = qa_service.answer(corpus=corpus, query=VISUAL_QUERY)
cached_visual_response = qa_service.answer(corpus=augmented_corpus, query=VISUAL_QUERY)
text_response = qa_service.answer(corpus=corpus, query=TEXT_QUERY)
corpus_counts = {}
for unit in corpus.units:
    corpus_counts[unit.unit_type.value] = corpus_counts.get(unit.unit_type.value, 0) + 1


In [ ]:
# inspect results
summary = {
    "retrieval_manifest": {
        "artifact_root": retrieval_manifest.artifact_root,
        "unit_count": retrieval_manifest.unit_count,
    },
    "corpus_counts": corpus_counts,
    "top_visual_hits": [
        {
            "unit_id": hit.unit.unit_id,
            "unit_type": hit.unit.unit_type.value,
            "page_span": [hit.unit.page_span.start_page, hit.unit.page_span.end_page],
            "score": hit.score,
        }
        for hit in visual_hits
    ],
    "visual_response": {
        "answer_mode": visual_response.answer_mode,
        "answer": visual_response.answer,
        "citations": [
            {
                "unit_id": citation.unit_id,
                "page_label": citation.page_label,
            }
            for citation in visual_response.citations
        ],
    },
    "cached_visual_response": {
        "answer_mode": cached_visual_response.answer_mode,
        "answer": cached_visual_response.answer,
        "citations": [
            {
                "unit_id": citation.unit_id,
                "page_label": citation.page_label,
            }
            for citation in cached_visual_response.citations
        ],
    },
    "text_response": {
        "answer_mode": text_response.answer_mode,
        "answer": text_response.answer,
        "citations": [
            {
                "unit_id": citation.unit_id,
                "page_label": citation.page_label,
                "page_span": [citation.page_span.start_page, citation.page_span.end_page],
                "quote": citation.quote,
            }
            for citation in text_response.citations
        ],
    },
}
print(json.dumps(summary, indent=2, ensure_ascii=True))


### Known Limitations
- The notebook uses a synthetic fixture bundle to keep execution deterministic and lightweight.
- Retrieval hits and evidence remain the production boundary for this phase; the QA adapter shown here is downstream convenience behavior.
- Live multimodal enrichment is not exercised here; the cached-attachment path is used instead.
- OpenRouter multimodal capability probes can now be treated as inconclusive in the LiteLLM adapter. LiteLLM cookbook vision calls should keep the `openrouter/...` alias without forcing `custom_llm_provider='openrouter'`, while direct OpenRouter `ChatOpenAI` calls should use the provider model ID (for example `google/...`).
- Query planning resolves `last page` dynamically inside `RetrievalService`, not in the raw `QueryPlan` model.
